# 05 · Performance Profiling & Optimisation
Measures how **caching**, **partitioning**, **column pruning** and **Parquet** change the runtime of a realistic analytics workload, inspects **physical plans**, and documents how to monitor jobs in the **Spark UI**.

Benchmarks use a warm-up run and report the **median of repeated runs** so one-off JVM/start-up effects do not distort results.

In [ ]:
# ── Environment & Spark session (shared by every notebook) ─────────────────────
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/app-store-analytics-spark"
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

# Raw CSV location (override with the APPSTORE_CSV environment variable)
DATA_PATH = os.environ.get("APPSTORE_CSV", os.path.join(PROJECT_DIR, "data", "raw", "apple_appstore_apps.csv"))
PROCESSED_PATH = os.path.join(PROJECT_DIR, "data", "processed", "apps_processed.parquet")
FIG_DIR = os.path.join(PROJECT_DIR, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
         .appName("AppStoreAnalytics")
         .config("spark.sql.adaptive.enabled", "true")                     # Adaptive Query Execution
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")  # merge small shuffle partitions
         .config("spark.sql.adaptive.skewJoin.enabled", "true")            # split skewed partitions
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | Spark UI: {spark.sparkContext.uiWebUrl}")

## 1. Open the Spark UI

In [ ]:
print("Spark UI:", spark.sparkContext.uiWebUrl)
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(4040, path="/jobs/")   # opens the Spark UI in a new tab

In the UI, check:
- **Jobs / Stages:** stage duration, number of tasks, shuffle read/write size;
- **Storage:** cached DataFrames, fraction in memory, size;
- **SQL / DataFrame:** the physical plan of each query and AQE re-optimisations (coalesced partitions, skew handling).

## 2. Benchmark helper and workload

In [ ]:
import time, statistics
import pandas as pd

def benchmark(fn, runs=5, label=""):
    fn()                                   # warm-up
    times = []
    for _ in range(runs):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    med = statistics.median(times)
    print(f"{label:<40} median {med:.3f}s  (min {min(times):.3f}s, max {max(times):.3f}s)")
    return med

def workload(d):
    """Typical analysis session: three aggregations + one window ranking."""
    d.groupBy("primary_genre").agg(F.avg("user_rating"), F.sum("rating_count")).collect()
    d.groupBy("price_category", "is_free").agg(F.count("*"), F.avg("rating_count")).collect()
    d.groupBy("size_category").agg(F.expr("percentile_approx(rating_count, 0.5)")).collect()
    w = Window.partitionBy("primary_genre").orderBy(F.desc("popularity_score"))
    d.withColumn("r", F.row_number().over(w)).filter("r <= 5").count()

## 3. CSV vs Parquet, with and without caching

In [ ]:
csv_df = (spark.read.option("header", "true").option("multiLine", "true").option("escape", '"').csv(DATA_PATH))
parquet_df = spark.read.parquet(PROCESSED_PATH)
results = {}

# Storage format
results["Full scan - raw CSV"] = benchmark(lambda: csv_df.count(), label="Full scan - raw CSV")
results["Full scan - Parquet"] = benchmark(lambda: parquet_df.count(), label="Full scan - Parquet")

# Cumulative optimisation steps on the same multi-query workload
spark.conf.set("spark.sql.shuffle.partitions", "200")          # Spark default
results["Workload 1: baseline (all cols, no cache, 200 shuffle partitions)"] = benchmark(
    lambda: workload(parquet_df), label="1 baseline")

pruned = parquet_df.select("primary_genre", "user_rating", "rating_count", "price_category",
                           "is_free", "size_category", "popularity_score")
results["Workload 2: + column pruning"] = benchmark(lambda: workload(pruned), label="2 + column pruning")

cached = pruned.cache(); cached.count()                         # materialise the cache once
results["Workload 3: + caching"] = benchmark(lambda: workload(cached), label="3 + caching")

tuned = str(max(8, 2 * spark.sparkContext.defaultParallelism))  # ~2x cores for a small local cluster
spark.conf.set("spark.sql.shuffle.partitions", tuned)
results[f"Workload 4: + shuffle partitions = {tuned}"] = benchmark(lambda: workload(cached), label="4 + shuffle tuning")

res = pd.Series(results, name="median_seconds").to_frame()
base = results["Workload 1: baseline (all cols, no cache, 200 shuffle partitions)"]
res["speed-up vs baseline"] = (base / res["median_seconds"]).round(2)
res.loc[res.index.str.startswith("Full scan"), "speed-up vs baseline"] = None
res

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
res["median_seconds"].sort_values().plot.barh(ax=ax, color="#2E5AAC")
ax.set_xlabel("median seconds (lower is better)"); ax.set_title("Benchmark results")
plt.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "05_benchmark.png"), dpi=150); plt.show()

> **Reading the results honestly:** caching only pays off when the **same data is reused by several queries**, which is why a multi-query workload is benchmarked. On small data, too many shuffle partitions create hundreds of tiny tasks whose scheduling overhead dominates, so tuning `spark.sql.shuffle.partitions` matters as much as caching. Explicit `repartition()` adds an extra shuffle and is only worth it when the partitioned data is reused many times. Report the numbers you observe.

## 4. Physical plans

In [ ]:
w = Window.partitionBy("primary_genre").orderBy(F.desc("popularity_score"))
print("=== Uncached, all columns ===")
parquet_df.withColumn("r", F.row_number().over(w)).explain(mode="formatted")
print("\n=== Pruned + cached ===")
cached.withColumn("r", F.row_number().over(w)).explain(mode="formatted")

Look for: `InMemoryTableScan` (cache hit), `ReadSchema` listing only the needed columns (pruning), `Exchange hashpartitioning(primary_genre)` (shuffle that pre-partitioning can avoid), and `AdaptiveSparkPlan` (AQE).

## 5. Broadcast join for small lookup tables

In [ ]:
genre_lookup = cached.groupBy("primary_genre").agg(F.avg("user_rating").alias("genre_avg_rating"))
joined = cached.join(F.broadcast(genre_lookup), "primary_genre") \
               .withColumn("rating_vs_genre", F.col("user_rating") - F.col("genre_avg_rating"))
joined.explain(mode="formatted")   # expect BroadcastHashJoin instead of SortMergeJoin
cached.unpersist()

## 6. Scaling plan for 50x data
| Bottleneck | Strategy |
|---|---|
| Memory | larger executors, `MEMORY_AND_DISK` persistence, avoid `toPandas()` on full tables |
| Shuffle / network | partition by `primary_genre`, broadcast small dimensions, tune `spark.sql.shuffle.partitions` (≈ 2–4 × total cores) |
| Skew | AQE skew-join handling; salting for dominant genres (e.g. Games) |
| Storage | Parquet with Snappy compression, partitioned by genre; incremental loads by update date |
| Computation | `percentile_approx` / `approxQuantile` instead of exact quantiles; cache only reused tables |
| Monitoring | Spark UI stage timelines, shuffle metrics and spill; event logs + History Server |